# Análisis del Mercado de Compras Públicas en Chile (2019–2025)

**Autor:** Guillermo Fuentes Jaque — Científico de Datos Geoespaciales  
**Repositorio:** [djwillichile/chile-public-procurement-analysis](https://github.com/djwillichile/chile-public-procurement-analysis)  
**Datos:** Mercado Público / ChileCompra — datos abiertos 2019–2025

---

## 1. Introducción

Este informe presenta un análisis exhaustivo del sistema de compras públicas de Chile, utilizando datos abiertos de **Mercado Público (ChileCompra)**. El objetivo central es responder la siguiente pregunta:

> **¿Qué sectores de productos o servicios muestran mayor crecimiento en la demanda del Estado chileno y representan oportunidades de negocio para proveedores?**

El análisis abarca:
- **Evolución del gasto público:** Serie temporal del gasto total y número de licitaciones (2019–2025).
- **Estacionalidad:** Patrones de gasto por mes y su repetición a lo largo de los años.
- **Análisis sectorial y regional:** Desglose del gasto por categoría de licitación y región geográfica.
- **Concentración de mercado:** Análisis de Pareto sobre proveedores adjudicados.
- **Identificación de oportunidades:** Índice de oportunidad de mercado que combina gasto, crecimiento y competencia.
- **Modelamiento predictivo:** Pronóstico de demanda (2025–2028) con modelos Prophet y XGBoost.
- **Trazabilidad del gasto:** Diagrama Sankey interactivo del flujo Sector → Categoría → Región.


## 2. Metodología

El proyecto sigue un pipeline de ciencia de datos reproducible de 6 etapas:

| Etapa | Script | Descripción |
|-------|--------|-------------|
| 1. Descarga | `src/download_bulk.py` | Descarga de archivos ZIP mensuales desde Azure Blob (ChileCompra), 74 archivos, 2019–2025 |
| 2. ETL Streaming | `src/process_streaming.py` | Procesamiento en streaming de >15 M registros (>30 GB CSV) → 6 Parquets agregados |
| 3. Limpieza | `src/clean_data.py` | Normalización de texto, fechas y montos; mapeo de 26 regiones; eliminación de outliers |
| 4. Features | `src/feature_engineering.py` | Métricas de crecimiento anual, competencia promedio e Índice de Oportunidad de Mercado |
| 5. Modelos | `src/modeling.py` | Pronóstico con **Prophet** (series temporales) y **XGBoost** (regresión con features rezagados), evaluados con MAE y RMSE |
| 6. Visualización | `src/visualizations.py` / `src/sankey_api.py` | 12 figuras estáticas + Sankey interactivo desde la API de Mercado Público |

**Resultado:** 807.597 licitaciones únicas procesadas, almacenadas en formato Parquet y analizadas en este informe.


## 3. Análisis Exploratorio de Datos

### 3.1. Evolución del Gasto Total en Compras Públicas

La figura muestra la evolución mensual del **monto adjudicado total** (eje izquierdo) y el **número de licitaciones** (eje derecho) entre 2019 y 2025. Se observan dos patrones relevantes:

- **Estacionalidad anual pronunciada:** El gasto se acelera sistemáticamente hacia el último trimestre de cada año (Q4), coincidiendo con el cierre presupuestario del Estado.
- **Peak pandémico en 2020:** Se registra un pico anómalo de gasto, probablemente asociado a compras de emergencia sanitaria (insumos médicos, equipamiento hospitalario), que no se repite en años posteriores.
- **Estabilización post-2021:** A partir de 2021, el número de licitaciones mensuales se mantiene relativamente estable en torno a las 10.000–15.000 licitaciones/mes.


![Evolución del Gasto Total](figures/fig01_evolucion_gasto_total.png)

### 3.2. Gasto por Sector Económico (2019–2025)

El gráfico de barras apiladas descompone el gasto anual por **sector comprador**, evidenciando la evolución relativa de cada organismo del Estado:

- **Ministerio de Obras Públicas** es el mayor comprador constante (≈ 2.850–3.420 MM CLP/año), con crecimiento sostenido en infraestructura vial e hídrica.
- **Salud** registra el pico más alto en 2020 (5.100 MM CLP), triplicando prácticamente su gasto ordinario por las compras de emergencia pandémica (insumos, ventiladores, EPP). Tras 2020 retorna a niveles pre-pandemia (~2.700 MM/año).
- **Municipalidades** mantienen un gasto estable de 1.300–1.530 MM CLP/año, reflejando la inversión local en mantención de infraestructura y servicios comunitarios.
- **FF.AA. y Defensa** y **Educación** tienen participaciones similares (~700–870 MM CLP/año), enfocadas respectivamente en equipamiento militar y material educativo.

**El pico de 2020** (total ≈ 12.400 MM CLP, +44% vs 2019) es el evento más significativo de la serie, confirmando que la pandemia fue el mayor impulsor puntual del gasto público en la historia reciente del sistema ChileCompra.

![Gasto por Sector Económico 2019–2025](figures/fig02_gasto_anual_sector.png)

### 3.3. Patrón de Estacionalidad del Gasto Público

El gráfico superpone la evolución mensual de cada año (líneas grises) junto con el promedio histórico (línea azul), revelando con claridad el **patrón estacional recurrente**:

- **Q1 (Ene–Mar):** Gasto bajo — inicio de ejecución presupuestaria.
- **Q2–Q3 (Abr–Sep):** Nivel medio estable — operaciones corrientes del Estado.
- **Q4 (Oct–Dic):** Pico de gasto — aceleración por cierre de año presupuestario. Diciembre es consistentemente el mes de mayor gasto.

**Implicancia para proveedores:** El momento óptimo para presentar ofertas es entre agosto y octubre, anticipándose a la demanda de Q4.


![Patrón Estacional del Gasto Público — Chile 2019–2025](figures/fig09_estacionalidad.png)


### 3.4. Top Categorías de Licitación por Monto Adjudicado

El ranking de las 15 categorías con mayor monto total adjudicado confirma la **dominancia de la infraestructura y la construcción** en el gasto público chileno:

- **Servicios de Construcción y Mantenimiento** y **Obras** lideran con amplio margen, concentrando más del 40% del gasto total.
- **Obras MINVU** aparece como la tercera categoría, reflejando la inversión habitacional del Estado.
- Categorías como **Vehículos y Equipamiento** y **Equipamiento Médico** también presentan montos significativos.

Este análisis es el punto de partida para el Índice de Oportunidad de Mercado (sección 4), que complementa el gasto con variables de crecimiento y competencia.


![Top Categorías](figures/fig03_top_categorias.png)

### 3.5. Distribución Geográfica del Gasto Público

La distribución regional del gasto refleja tanto la densidad institucional como la inversión en infraestructura de cada zona del país:

- **Región Metropolitana:** Concentra la mayor parte del gasto, por ser la sede del Gobierno Central, los ministerios y la mayor cantidad de organismos compradores.
- **Coquimbo y Atacama:** Destacan con montos per cápita elevados, asociados a grandes proyectos de infraestructura minera, vial y de recursos hídricos en el norte semiárido.
- **Biobío y La Araucanía:** Relevantes por inversión en infraestructura social y habitacional.
- **Regiones extremas (Magallanes, Aysén, Arica):** Menor volumen total pero alta relevancia relativa por su aislamiento geográfico.


![Gasto por Región](figures/fig04_gasto_region.png)

### 3.6. Concentración de Proveedores — Análisis de Pareto

El análisis de Pareto sobre los **top 50 proveedores** por monto adjudicado total revela una alta concentración del mercado:

- Las **barras** muestran el monto adjudicado de cada proveedor en ranking descendente.
- La **línea naranja** indica el porcentaje acumulado del gasto total.
- La **línea punteada** marca el umbral del 80%.

**Hallazgo clave:** Un número muy reducido de proveedores concentra la mayor parte del gasto — fenómeno típico en mercados de infraestructura y construcción, donde las barreras de entrada técnicas y financieras son altas. Esto confirma la existencia de **nichos de baja competencia** que el Índice de Oportunidad (sección 4) ayuda a identificar.


![Concentración de Proveedores — Análisis de Pareto (Top 50)](figures/fig11_proveedores_concentracion.png)


### 3.7. Correlación Gasto Público vs Población por Región

El diagrama de dispersión (panel izquierdo) enfrenta el **gasto total en compras públicas** (eje Y) con la **población regional** (eje X, INE 2022). La línea punteada representa la proporcionalidad perfecta: regiones sobre esta línea reciben más recursos de los que su tamaño demográfico justificaría.

El gráfico de barras horizontales (panel derecho) cuantifica el **gasto per cápita** (CLP/habitante) para cada región:

- **Atacama y Coquimbo** encabezan el gasto per cápita (~3.200 y ~1.440 miles CLP/hab respectivamente), impulsadas por grandes contratos de infraestructura minera, vial y de recursos hídricos — no por su tamaño poblacional.
- **La Región Metropolitana**, pese a tener el mayor gasto absoluto (~6.100 MM CLP), se sitúa **bajo el promedio per cápita** nacional, evidenciando que su concentración es consecuencia de la densidad institucional más que de una sobreasignación relativa.
- **Magallanes y Aysén** superan el promedio per cápita debido a sus altos costos logísticos y de infraestructura en territorios extremos.
- **Biobío, La Araucanía y O'Higgins** se ubican bajo el promedio per cápita, sugiriendo potencial de mayor inversión en estas regiones con importante población rural.

**Conclusión:** El gasto público no sigue estrictamente la distribución poblacional — está fuertemente condicionado por los proyectos de inversión en infraestructura y recursos naturales.

![Gasto Público vs Población por Región](figures/fig15_scatter_gasto_poblacion.png)

## 4. Identificación de Oportunidades de Mercado

### 4.1. Heatmap: Evolución del Gasto por Categoría y Año

El heatmap muestra la **distribución relativa del gasto de cada categoría a lo largo del tiempo** (cada fila suma 100%). Permite detectar:

- **Categorías en crecimiento:** Aquellas cuyo porcentaje aumenta en años recientes (colores más intensos hacia 2024–2025).
- **Categorías en declive:** Aquellas que muestran reducción relativa post-2020.
- **Impacto COVID-2020:** En varias categorías se observa un peak de color en 2020 seguido de normalización.

![Heatmap: Gasto Relativo por Categoría y Año](figures/fig10_heatmap_categoria_anio.png)

### 4.2. Índice de Oportunidad de Mercado

Para priorizar categorías estratégicamente, se construyó un **Índice de Oportunidad de Mercado (IO)** que combina tres dimensiones mediante ponderación:

| Dimensión | Peso | Descripción |
|-----------|------|--------------|
| Gasto total | 40% | Volumen absoluto de adjudicaciones |
| Crecimiento anual | 30% | Variación % del gasto en el último año |
| Baja competencia | 30% | Inverso del promedio de oferentes |

Todas las dimensiones se normalizan entre 0 y 1 antes de combinarlas. Las categorías se clasifican en **Baja / Media / Alta / Muy Alta** oportunidad.

**Resultados principales:**
- **Obras** (IO=0.675, Alta): combina alto gasto histórico con crecimiento sostenido.
- **Vehículos y Equipamiento** (IO=0.590, Alta): crecimiento explosivo en adjudicaciones recientes.
- **Servicios Financieros** (IO=0.300, Media): baja competencia (2.1 oferentes promedio) — nicho accesible.


![Índice de Oportunidad](figures/fig06_indice_oportunidad.png)

## 5. Modelamiento Predictivo (2025–2028)

Se implementaron **dos modelos complementarios** para pronosticar el gasto total en compras públicas:

### Prophet (Meta)
Modelo de series temporales diseñado para capturar **tendencias y estacionalidades** complejas. Entrenado con datos 2019–2024, validado en 2025.
- Captura el patrón Q4 y la tendencia general con banda de confianza al 80%.
- La predicción muestra **estabilización del gasto** sin los peaks pandémicos, con estacionalidad anual mantenida.

### XGBoost (Gradient Boosting)
Modelo de regresión con **variables rezagadas** (1, 2, 3, 6 y 12 meses) y promedios móviles (3, 6, 12 meses). Entrenado hasta julio 2024, testeado en el segundo semestre 2024.
- Captura relaciones no lineales entre períodos de gasto.
- Complementa a Prophet en la detección de patrones de corto plazo.

**Evaluación:** Ambos modelos se evalúan con **MAE** (Error Absoluto Medio) y **RMSE** (Raíz del Error Cuadrático Medio). El horizonte de predicción cubre **2025 a 2028**.


![Forecast del Gasto Total](figures/fig07_forecast_total.png)

## 6. Trazabilidad del Gasto: Diagrama Sankey de Flujo de Inversión Pública

Se generó un **diagrama Sankey interactivo de tres niveles** que visualiza el flujo completo de inversión pública:

> **Sector Comprador → Categoría de Licitación → Región**

Generado con `src/sankey_api.py` a partir de los datos de la API de Mercado Público. La versión interactiva permite hacer hover, zoom y reorganizar nodos: **[Ver Sankey interactivo →](sankey.html)**

![Flujo de Inversión Pública: Sector → Categoría → Región](figures/fig14_sankey_flujo_interactivo.png)

### Observaciones del Sankey

**Sectores compradores** (eje izquierdo):
- El **Ministerio de Obras Públicas** es el mayor comprador con ~3.100 MM CLP, concentrado en Obras Públicas (1.900 MM) y Construcción y Mant. (1.200 MM).
- **Salud** es el segundo sector por gasto (~2.700 MM CLP), distribuido entre Construcción (1.500 MM), Vehículos y Equip. (750 MM) y Servicios Varios (450 MM).
- **Municipalidades** reparte su inversión equitativamente entre Obras Públicas (900 MM) y Construcción (600 MM).
- **FF.AA. y Defensa** concentra el 77% de su gasto en Vehículos y Equipamiento (650 de 850 MM totales), con inversión mínima en infraestructura.

**Destino regional** (eje derecho):
- La **R. Metropolitana** recibe el ~63% del gasto total (~6.100 MM CLP), siendo el destino dominante en todas las categorías, especialmente Construcción (2.750 MM).
- La **Macrozona Norte** es el segundo destino regional (~1.500 MM), principalmente por Obras Públicas (900 MM) financiadas por MOP y Municipalidades.
- La **Macrozona Sur** (~1.350 MM) recibe flujo balanceado de Construcción y Obras, reflejando inversión en infraestructura regional.
- **Servicios Varios** fluye casi exclusivamente hacia la R. Metropolitana (950 de 1.150 MM = 83%), confirmando la centralización de los servicios públicos.


## 7. Conclusiones y Limitaciones

### Conclusiones Principales

1. **Estacionalidad estructural:** El gasto público chileno tiene un patrón Q4 muy marcado y predecible. Los proveedores que anticipan sus ofertas en agosto–octubre tienen ventaja competitiva.

2. **Dominancia de infraestructura:** Servicios de Construcción y Mantenimiento y Obras concentran más del 40% del gasto total, siendo las categorías de mayor volumen absoluto.

3. **Oportunidades estratégicas:** El Índice de Oportunidad identifica **Obras** (IO=0.675) y **Vehículos y Equipamiento** (IO=0.590) como las áreas de mayor potencial, combinando alto gasto, crecimiento y competencia moderada. **Servicios Financieros** destaca por su baja competencia (2.1 oferentes promedio).

4. **Concentración geográfica y per cápita:** La R. Metropolitana centraliza el gasto absoluto, pero Atacama y Coquimbo lideran el gasto per cápita (~3.200 y ~1.440 miles CLP/hab), impulsadas por infraestructura y minería. Biobío, La Araucanía y O'Higgins reciben menos de lo que su peso poblacional sugeriría.

5. **Mercado altamente concentrado:** Un número reducido de proveedores captura la mayor parte del gasto (principio de Pareto), especialmente en construcción e infraestructura.

6. **Predicción estable 2025–2028:** Los modelos Prophet y XGBoost indican estabilización del gasto con estacionalidad mantenida, sin nuevos shocks equivalentes al de 2020.

### Limitaciones

- Los datos de la API pública pueden presentar desfases temporales de semanas a meses.
- El análisis excluye contrataciones directas por trato directo y compras menores a 3 UTM, que no se publican en la plataforma.
- Las categorías de Rubro (Rubro1) están definidas por ChileCompra y pueden cambiar entre períodos, dificultando la comparación de largo plazo.
- Los modelos predictivos asumen continuidad de patrones históricos; shocks exógenos (emergencias, cambios de política) no son capturados.

### Próximos Pasos

- **Análisis de Redes:** Grafo organismo–proveedor para detectar concentración y relaciones recurrentes.
- **Clasificador de Adjudicación:** Modelo que prediga la probabilidad de que un proveedor se adjudique una licitación específica.
- **Dashboard Interactivo:** Aplicación Streamlit/Dash para exploración dinámica de los datos.
- **Alertas en Tiempo Real:** Sistema de notificaciones para nuevas licitaciones en categorías de alto interés.
